In [1]:
!pip install fiftyone

  Using cached fiftyone-1.15.0-py3-none-any.whl.metadata (23 kB)
  Using cached argcomplete-3.6.3-py3-none-any.whl.metadata (16 kB)
  Using cached dacite-1.9.2-py3-none-any.whl.metadata (17 kB)
  Using cached ftfy-6.3.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached humanize-4.15.0-py3-none-any.whl.metadata (7.8 kB)
  Using cached hypercorn-0.18.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached mongoengine-0.29.3-py3-none-any.whl.metadata (7.0 kB)
  Using cached motor-3.6.1-py3-none-any.whl.metadata (21 kB)
  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached pprintpp-0.4.0-py2.py3-none-any.whl.metadata (7.9 kB)
  Using cached protobuf-6.33.5-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
  Using cached pydash-8.0.6-py3-none-any.whl.metadata (3.4 kB)
  Using cached pymongo-4.9.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (22 kB)
  Using cached sseclient_py-1.9.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached starlette-0

In [2]:
import fiftyone as fo 
dataset = fo.Dataset.from_dir(dataset_dir="CarDD",
                              dataset_type=fo.types.FiftyOneDataset,
                             )
print(dataset) 
print(dataset.first())

Importing samples...
 100% |███████████████| 2816/2816 [177.5ms elapsed, 0s remaining, 16.2K samples/s] 
Migrating dataset '2026.05.15.16.09.00.527838' to v1.15.0
Name:        2026.05.15.16.09.00.527838
Media type:  image
Num samples: 2816
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    detections:       fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    segmentations:    fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    coco_id:          fiftyone.core.fields.IntField
<Sample: {
    'id': '686be771e1d7135d782c77e9

### To Yolo Structure

In [3]:
import shutil
import fiftyone as fo
import fiftyone.utils.random as four

classes = sorted(dataset.distinct("detections.detections.label"))
print(classes)

four.random_split(
    dataset,
    {"train": 0.7, "val": 0.2, "test": 0.1},
    seed=42
)

shutil.rmtree("CarDD_YOLO", ignore_errors=True)

for split in ["train", "val", "test"]:
    view = dataset.match_tags(split)

    view.export(
        export_dir="CarDD_YOLO",
        dataset_type=fo.types.YOLOv5Dataset,
        label_field="detections",
        split=split,
        classes=classes,
    )

['crack', 'dent', 'glass shatter', 'lamp broken', 'scratch', 'tire flat']
 100% |███████████████| 1971/1971 [25.0s elapsed, 0s remaining, 77.7 samples/s]       
Directory 'CarDD_YOLO' already exists; export will be merged with existing files
 100% |█████████████████| 563/563 [6.8s elapsed, 0s remaining, 83.0 samples/s]      
Directory 'CarDD_YOLO' already exists; export will be merged with existing files
 100% |█████████████████| 282/282 [3.9s elapsed, 0s remaining, 92.9 samples/s]      


In [4]:
from collections import Counter

for split in ["train", "val", "test"]:
    view = dataset.match_tags(split)
    c = Counter()

    for sample in view:
        if sample.detections:
            for d in sample.detections.detections:
                c[d.label] += 1

    print(split)
    print("images:", len(view))
    print(c)

train
images: 1971
Counter({'scratch': 1821, 'dent': 1211, 'crack': 440, 'glass shatter': 340, 'lamp broken': 328, 'tire flat': 149})
val
images: 563
Counter({'scratch': 500, 'dent': 391, 'crack': 133, 'lamp broken': 109, 'glass shatter': 87, 'tire flat': 59})
test
images: 282
Counter({'scratch': 239, 'dent': 204, 'crack': 78, 'lamp broken': 57, 'glass shatter': 48, 'tire flat': 17})


### To COCO Structure

In [25]:
import shutil
import fiftyone as fo
import fiftyone.utils.random as four

dataset = fo.Dataset.from_dir(
    dataset_dir="CarDD",
    dataset_type=fo.types.FiftyOneDataset,
)

classes = sorted(dataset.distinct("detections.detections.label"))
print(classes)

four.random_split(
    dataset,
    {"train": 0.7, "valid": 0.2, "test": 0.1},
    seed=42
)

shutil.rmtree("CarDD_RFDETR", ignore_errors=True)

for split in ["train", "valid", "test"]:
    view = dataset.match_tags(split)

    view.export(
        export_dir=f"CarDD_RFDETR/{split}",
        dataset_type=fo.types.COCODetectionDataset,
        label_field="detections",
        classes=classes,
        export_media=True,
    )

from pathlib import Path
import shutil

root = Path("CarDD_RFDETR")

for split in ["train", "valid", "test"]:
    split_dir = root / split
    data_dir = split_dir / "data"
    labels_path = split_dir / "labels.json"

    if labels_path.exists():
        shutil.move(str(labels_path), str(split_dir / "_annotations.coco.json"))

    if data_dir.exists():
        for img_path in data_dir.iterdir():
            if img_path.is_file():
                shutil.move(str(img_path), str(split_dir / img_path.name))

        shutil.rmtree(data_dir)

Importing samples...
 100% |███████████████| 2816/2816 [113.6ms elapsed, 0s remaining, 25.2K samples/s]  
Migrating dataset '2026.05.10.05.43.37.236833' to v1.15.0
['crack', 'dent', 'glass shatter', 'lamp broken', 'scratch', 'tire flat']
 100% |███████████████| 1971/1971 [6.2s elapsed, 0s remaining, 331.9 samples/s]       
 100% |█████████████████| 563/563 [1.7s elapsed, 0s remaining, 328.3 samples/s]         
 100% |█████████████████| 282/282 [876.7ms elapsed, 0s remaining, 321.6 samples/s]      


In [31]:
import os

print(len(os.listdir("CarDD/data")))

print("CarD_YOLO")
print(len(os.listdir("CarDD_YOLO/images/train")))
print(len(os.listdir("CarDD_YOLO/images/val")))
print(len(os.listdir("CarDD_YOLO/images/test")))

print("CarDD_RFDETR")
print(len(os.listdir("CarDD_RFDETR/train")))
print(len(os.listdir("CarDD_RFDETR/valid")))
print(len(os.listdir("CarDD_RFDETR/test")))

2816
CarD_YOLO
1971
563
282
CarDD_RFDETR
1972
564
283
